# Cosmic Engine — Shot-Graph / Stargate Sequence (manifest-driven)

This notebook exercises the architecture on branch `claude/keen-maxwell-sAB8n`:

- `scene/` — ShotGraph + Shot + Transition + Palette manifest
- `render/core.py` — Renderer ABC + name registry
- `render/noise.py` — procedural detail toolkit (footprint-gated fBM, ridged, domain-warp, worley, curl). The system-wide anti-pixelation invariant.
- `render/adapters.py` — wraps gas-giant, volumetric gas giant, Kerr black hole, slit-scan tunnel, exotic physics, **spiral galaxy, diffuse nebula, saturn-class, stellar surface**
- `render/volumetric_gas_giant.py` — Juno-quality Jovian renderer
- `render/saturn_class.py` — Saturn-class ringed gas giant: paler bands, full ring system (C/B/Cassini/A/Encke/F), bi-directional ring/planet shadows, hexagonal polar vortex
- `render/spiral_galaxy.py` — M51-class volumetric spiral-galaxy renderer
- `render/diffuse_nebula.py` — six-topology nebula renderer (pillars / Crab / Helix / Veil / Orion / Pleiades)
- `render/stellar_surface.py` — Sun-class / M-dwarf / O-star renderer; limb darkening, Worley granulation, parametric sunspots, Halpha prominences
- `render/kerr.py` — Kerr ray-tracer
- `render/io.py` — ACES filmic + sRGB tonemap, EXR/MP4 writers
- `director/continuity.py` — histogram-matched crossfade + smoothstep blend
- `director/graph.py` — overlapping-transition graph executor
- `studio/` — artifact store, hand-coded metrics critic, mutation space, producer, session loop

**Runtime:** set the Colab runtime to **GPU** (T4 is fine).

End-to-end pipeline this notebook drives:

1. Install deps and clone the branch.
2. Run the CPU test suite to verify the architecture imports cleanly.
3. Load `scene/examples/jupiter_to_stargate.json` (Jovian approach → Kerr anomaly → slit-scan tunnel).
4. Render it to MP4 via `GraphRunner` and display inline.
5. Tweak the manifest live (palette, Kerr spin, transition kind) and re-render.
6. Hand-author a fresh ShotGraph from scratch.
7. Demo the new `volumetric_gas_giant` renderer side-by-side against the legacy `gas_giant`, then drive the three-shot Juno-style approach manifest.
8. Render the M51 Whirlpool — face-on push-in, arm close-up, then the companion + tidal bridge.
9. Render the nebula trinity — Pillars of Creation → Crab SNR → Helix planetary nebula.
10. Render the Saturn voyage — oblique full disc → ring skim → north-pole hexagon.
11. Render the stellar trio — Sun-like active region → M-dwarf with starspots → hot O-star.
12. Use the studio artifact store + metrics critic to score a render and show keyframes.
13. Close the loop: explore the parameter space with a Producer + Session.

## 1. Install dependencies

In [ ]:
!pip -q install taichi 'imageio[ffmpeg,pyav]' av numpy
!apt-get -qq install -y ffmpeg > /dev/null
import taichi, imageio, av, numpy as np
print('taichi', taichi.__version__, '| imageio', imageio.__version__, '| av', av.__version__, '| numpy', np.__version__)

## 2. Clone the branch

Uses a shallow clone of the feature branch directly under `/content/cosmic_engine`. Re-running the cell pulls the latest commit on the same branch.

In [ ]:
import os, subprocess, sys

REPO_URL  = 'https://github.com/pmcray/cosmic_engine.git'
BRANCH    = 'claude/keen-maxwell-sAB8n'
REPO_DIR  = '/content/cosmic_engine'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', BRANCH], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

head = subprocess.check_output(['git', '-C', REPO_DIR, 'log', '-1', '--oneline']).decode().strip()
print('HEAD:', head)

# Evict cached cosmic_engine modules so re-running cells after a git pull
# picks up the freshly cloned source instead of the previous import's bytecode.
# Jupyter does not invalidate sys.modules when files on disk change, which is
# what produces the misleading "old traceback through new source" symptom.
_PURGE_PREFIXES = ('render', 'director', 'scene', 'infinite_director', 'encounters', 'physics')
_purged = [m for m in list(sys.modules) if m.startswith(_PURGE_PREFIXES)]
for _m in _purged:
    del sys.modules[_m]
if _purged:
    print(f'cleared {len(_purged)} cached cosmic_engine module(s); next import will re-read disk')

## 3. Smoke test the architecture (CPU only)

All eight tests should pass. They cover manifest round-trip, camera interpolation, the palette registry, ACES tone-mapping behaviour, the histogram-matched crossfade, the transition engine, the example manifest loader, and an end-to-end `GraphRunner` run on the trivial constant renderer. **No GPU touched yet.**

In [ ]:
!python -m tests.test_manifest

## 4. Inspect the example manifest

In [ ]:
from scene.manifest import load_manifest
from scene.palette import PALETTES

MANIFEST_PATH = 'scene/examples/jupiter_to_stargate.json'
graph = load_manifest(MANIFEST_PATH)

print(f'Title: {graph.title}')
print('Shots:')
for s in graph.shots:
    print(f'  {s.id:18s} renderer={s.renderer:18s} dur={s.duration_frames:>3d}f @ {s.fps}fps  res={s.resolution} palette={s.palette.name}')
print('Transitions:')
for t in graph.transitions:
    print(f'  {t.from_shot:18s} -> {t.to_shot:18s} kind={t.kind:10s} dur={t.duration_frames}f params={t.params}')
print('\nAvailable palettes:', sorted(PALETTES.keys()))

## 5. Fast-mode override

Full 1920×1080 × 336 frames takes several minutes per shot on a T4 and the first Kerr render also pays a one-off Taichi JIT cost. The cell below downscales the in-memory graph for a sanity render that finishes in roughly a minute. Set `FAST_MODE = False` for the full version.

In [ ]:
FAST_MODE = True

if FAST_MODE:
    for s in graph.shots:
        s.resolution = (640, 360)
        s.duration_frames = max(24, s.duration_frames // 4)
    for t in graph.transitions:
        t.duration_frames = max(6, t.duration_frames // 3)

print('Effective render plan:')
for s in graph.shots:
    print(f'  {s.id:18s} {s.duration_frames:>3d}f  {s.resolution}')
for t in graph.transitions:
    print(f'  {t.from_shot} -> {t.to_shot:18s} {t.kind:10s} {t.duration_frames}f')

## 6. Render the example manifest end-to-end

First run also JIT-compiles every Taichi kernel. Expect a one-time 20–40 s warm-up before frames start rolling.

In [ ]:
import time, os
import render.adapters  # registers gas_giant, kerr_black_hole, slitscan_tunnel, exotic_physics
from director.graph import GraphRunner

os.makedirs('outputs', exist_ok=True)
out_path = 'outputs/jupiter_to_stargate.mp4'

t0 = time.time()
GraphRunner(graph, out_path).run()
dt = time.time() - t0
print(f'wrote {out_path} ({os.path.getsize(out_path) / 1024:.1f} KiB) in {dt:.1f}s')

## 7. Inline playback

In [ ]:
from IPython.display import HTML
from base64 import b64encode

def show_mp4(path, width=720):
    data = open(path, 'rb').read()
    b64  = b64encode(data).decode()
    return HTML(
        f'<video width={width} controls autoplay loop muted playsinline>'
        f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'
    )

show_mp4(out_path)

## 8. Tweak the manifest live

The `graph` is just dataclasses — mutate it in memory and re-render. Here we swap to the JWST NIRCam palette and spin the Kerr black hole up to near-extremal `a/M = 0.99`.

In [ ]:
for s in graph.shots:
    s.palette.name = 'jwst_nircam'

kerr = graph.shot_by_id('kerr_anomaly')
kerr.params['spin']            = 0.99
kerr.params['inclination_deg'] = 87.0
kerr.params['disk_outer']      = 18.0

out_path2 = 'outputs/jupiter_to_stargate_jwst_spin99.mp4'
GraphRunner(graph, out_path2).run()
show_mp4(out_path2)

## 9. Hand-author a fresh ShotGraph

Build a two-shot sequence — near-extremal Kerr followed by a slit-scan tunnel — straight from Python, using the slit-scan family for the transition.

In [ ]:
from scene.manifest import Camera, PaletteRef, Shot, ShotGraph, Transition

my_graph = ShotGraph(
    title='kerr_into_stargate',
    shots=[
        Shot(
            id='kerr',
            renderer='kerr_black_hole',
            params={'spin': 0.95, 'inclination_deg': 85.0, 'disk_outer': 16.0, 'steps': 240},
            palette=PaletteRef(name='hubble_sii_ha_oiii'),
            duration_frames=48,
            fps=24,
            resolution=(640, 360),
            motion_hint='approach',
        ),
        Shot(
            id='tunnel',
            renderer='slitscan_tunnel',
            params={'time_scale': 0.06},
            palette=PaletteRef(name='trumbull_2001', intensity=1.2),
            duration_frames=64,
            fps=24,
            resolution=(640, 360),
            motion_hint='tunnel',
        ),
    ],
    transitions=[
        Transition(from_shot='kerr', to_shot='tunnel', kind='slitscan', duration_frames=18, params={'intensity': 1.4}),
    ],
)
my_graph.validate()

out_path3 = 'outputs/custom_kerr_into_stargate.mp4'
GraphRunner(my_graph, out_path3).run()
show_mp4(out_path3)

## 10. Juno-quality gas giant: `volumetric_gas_giant`

A new renderer (`volumetric_gas_giant`) targets ~95% of Juno-quality Jupiter imagery while leaving the legacy `gas_giant` untouched for comparison.

What's different:

- **Atmospheric ray-march** on a thin Rayleigh+Mie shell — limb darkening, twilight terminator, and forward-scatter glow all fall out of the same integral.
- **Three-layer cloud composite** (NH₃ ice / NH₄SH / H₂O) baked into the surface color via Beer–Lambert. Rare windows where all three layers thin become the deep-blue "5-micron hotspots."
- **Realistic Jovian banding** from the SPR → SSTZ → … → NPR zonal-wind table. Each row drifts with the observed jet velocity, so bands shear past each other independently.
- **Storms placed at correct latitudes**: Great Red Spot (-22.5°), white ovals on the STZ, brown barges on the NEB, and the Juno polar-cyclone pattern (1 central + 8 surrounding at lat = ±83°).
- **Kelvin–Helmholtz rolls** at zone/belt boundaries.

The cell renders two things: a same-framing side-by-side of legacy vs new, then the full `jupiter_juno_approach.json` manifest (perijove push-in → Great Red Spot close-up → north-pole cyclones).

In [ ]:
import os, time
from base64 import b64encode
from IPython.display import HTML, display

from scene.manifest import PaletteRef, Shot, ShotGraph, load_manifest
import render.adapters  # ensures gas_giant + volumetric_gas_giant are registered

def _video_html(path, width=420):
    data = open(path, 'rb').read()
    b64 = b64encode(data).decode()
    return (
        f'<video width={width} controls autoplay loop muted playsinline>'
        f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'
    )

os.makedirs('outputs', exist_ok=True)

# A) Side-by-side: same camera framing, both renderers.
def _one_shot(renderer, params, label):
    return ShotGraph(
        title=label,
        shots=[Shot(
            id='shot',
            renderer=renderer,
            params=params,
            palette=PaletteRef('trumbull_2001'),
            duration_frames=24,
            fps=24,
            resolution=(512, 512),
        )],
    )

old_params = {'prewarm_frames': 30, 'pan': 0.0, 'pan_rate': 0.3,
              'tilt': -0.2, 'rings': 0, 'samples': 2}
new_params = {'march_steps': 24, 'samples': 2,
              'sun_dir': [0.55, 0.18, 0.81], 'grs_lon': 100.0, 'seed': 42}

for label, graph in [('cmp_old', _one_shot('gas_giant', old_params, 'cmp_old')),
                     ('cmp_new', _one_shot('volumetric_gas_giant', new_params, 'cmp_new'))]:
    out = f'outputs/{label}.mp4'
    t0 = time.time()
    GraphRunner(graph, out).run()
    print(f'wrote {out} in {time.time() - t0:.1f}s')

display(HTML(
    f'<div style="display:flex; gap:16px; align-items:flex-start;">'
    f'<div><strong>legacy <code>gas_giant</code></strong><br/>{_video_html("outputs/cmp_old.mp4")}</div>'
    f'<div><strong>new <code>volumetric_gas_giant</code></strong><br/>{_video_html("outputs/cmp_new.mp4")}</div>'
    f'</div>'
))

# B) The full Juno-approach example: perijove push-in, GRS close-up, polar cyclones.
juno_graph = load_manifest('scene/examples/jupiter_juno_approach.json')
# Fast-mode downscale so the cell finishes in a couple minutes on a T4.
for s in juno_graph.shots:
    s.resolution = (640, 360)
    s.duration_frames = max(24, s.duration_frames // 3)
for tr in juno_graph.transitions:
    tr.duration_frames = max(4, tr.duration_frames // 3)

t0 = time.time()
GraphRunner(juno_graph, 'outputs/juno_approach.mp4').run()
print(f'\nJuno approach: wrote outputs/juno_approach.mp4 in {time.time() - t0:.1f}s')
display(HTML(_video_html('outputs/juno_approach.mp4', width=720)))

## 11. M51 Whirlpool — volumetric `spiral_galaxy`

A new renderer that builds a face-on (or tilted) spiral galaxy entirely from parametric structures and procedural noise — no baked galaxy texture anywhere. Detail resolves at every zoom level via the footprint-aware fBM primitives in `render/noise.py` (the system-wide anti-pixelation invariant).

What it composes:

- **Logarithmic spiral arms**: density modulation `1 + arm_strength · cos(N·θ − k·log r + φ)` with `k = N / tan(pitch)`. Two arms for grand-design (M51-like).
- **Sersic n=4 bulge** (oblate ellipsoid) for the warm-yellow core of old stars.
- **Exponential disk** (radial × vertical) with arm modulation × `domain_warp_fbm` swirl perturbation.
- **`ridged_fbm_3d` dust lanes** along the inside edge of arms, confined to the disk mid-plane.
- **`worley_3d` HII regions** — pink clumps along the arm crests.
- **Sparse extended halo** outside the disk.
- **M51 companion (NGC 5195)**: parameterized; second Sersic bulge plus an inverse-square tidal-bridge density along the segment between the two centers.
- **Per-channel transmittance**: dust extinction reddens the disk naturally (blue absorbed 1.7× more than red).
- **Doppler tint**: at non-zero inclination, one arm side warms and the other cools.

The example manifest `scene/examples/m51_whirlpool.json` drives three shots: face-on grand-design (isolated, push-in), an arm close-up at 30° inclination (isolated, narrowband palette), and the companion+bridge framing (the namesake M51-ness, JWST palette).

Lord Rosse's 1845 sketch first resolved spiral structure in M51 using the Leviathan of Parsonstown. This renderer is built to honour both ends of the M51 reference spectrum: that historical sketch and the modern JWST mid-infrared image.

In [ ]:
import os, time
from base64 import b64encode
from IPython.display import HTML, display

from scene.manifest import load_manifest
import render.adapters  # registers spiral_galaxy
from director.graph import GraphRunner

os.makedirs('outputs', exist_ok=True)

m51_graph = load_manifest('scene/examples/m51_whirlpool.json')

# Fast-mode downsample so the cell finishes in a few minutes on a T4.
# Comment this block out for the full-quality 1280x720 master.
for s in m51_graph.shots:
    s.resolution = (640, 360)
    s.duration_frames = max(24, s.duration_frames // 3)
    # Also lighten the ray-march per shot for proxy speed.
    s.params['march_steps'] = max(36, s.params.get('march_steps', 72) // 2)
for tr in m51_graph.transitions:
    tr.duration_frames = max(4, tr.duration_frames // 3)

t0 = time.time()
GraphRunner(m51_graph, 'outputs/m51_whirlpool.mp4').run()
print(f'wrote outputs/m51_whirlpool.mp4 in {time.time() - t0:.1f}s')


def _video_html(path, width=720):
    data = open(path, 'rb').read()
    b64 = b64encode(data).decode()
    return (
        f'<video width={width} controls autoplay loop muted playsinline>'
        f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'
    )


display(HTML(_video_html('outputs/m51_whirlpool.mp4', width=820)))

## 12. Diffuse nebula — six topologies, one renderer

`diffuse_nebula` shares the volumetric emission kernel with the spiral galaxy and switches a `topology` parameter to produce six iconic forms — each a parameterization, not a new renderer:

| Topology | What it is | What you'll see |
|---|---|---|
| `pillars` | Pillars of Creation (M16) | Vertical columns eroded from above, pink HII rims, dust-heavy interiors silhouetted against the glow |
| `crab` | Crab supernova remnant (M1) | Expanding ovoid shell of red filaments with bluish synchrotron interior + central pulsar |
| `helix` | Helix planetary nebula (NGC 7293) | Green-cyan OIII torus, warm Halpha bipolar lobes, central white dwarf |
| `veil` | Veil SNR | Thin shock-front shell with twin Doppler layers (approaching blue, receding red) |
| `orion` | Orion HII region (M42) | Turbulent pink-purple glow with embedded young-star cluster and dust silhouettes |
| `pleiades` | Pleiades reflection nebula | Diffuse blue dust cloud scattering foreground starlight |

Same architectural invariants as the galaxy: no baked texture, per-channel transmittance so dust extinction reddens correctly, and every screen-space noise call is footprint-gated via `render/noise.py` — so the same renderer holds up at Hubble-wide-field and JWST-close-up.

The example manifest `scene/examples/nebula_trinity.json` drives the three most iconic forms in sequence: Pillars push-in → Crab drift → Helix slow pan.

In [ ]:
import os, time
from base64 import b64encode
from IPython.display import HTML, display

from scene.manifest import load_manifest
import render.adapters  # registers diffuse_nebula
from director.graph import GraphRunner

os.makedirs('outputs', exist_ok=True)

neb_graph = load_manifest('scene/examples/nebula_trinity.json')

# Fast-mode downsample so the cell finishes in a few minutes on a T4.
# Comment this block out for the full-quality 1280x720 master.
for s in neb_graph.shots:
    s.resolution = (640, 360)
    s.duration_frames = max(24, s.duration_frames // 3)
    s.params['march_steps'] = max(40, s.params.get('march_steps', 80) // 2)
for tr in neb_graph.transitions:
    tr.duration_frames = max(4, tr.duration_frames // 3)

t0 = time.time()
GraphRunner(neb_graph, 'outputs/nebula_trinity.mp4').run()
print(f'wrote outputs/nebula_trinity.mp4 in {time.time() - t0:.1f}s')

display(HTML(_video_html('outputs/nebula_trinity.mp4', width=820)))

## 13. Saturn-class — rings, hexagon, shadow casting

`saturn_class` extends the volumetric gas-giant pattern with:

- **Saturn-tuned band table** — paler creams and golds; a single dramatic equatorial jet (~470 m/s, several times Jupiter's)
- **Parametric ring system**: C ring → B ring → Cassini Division → A ring (with Encke gap) → F ringlet, each with its own inner/outer radius, density, and color
- **Bi-directional shadow casting**: ring shadows fall on the planet AND the planet shadow falls on the rings — both via simple ray-sphere / ray-plane tests inside the kernel
- **Hexagonal north-polar vortex**: six-fold cosine in longitude × Gaussian envelope around `hex_lat ≈ 78°` (the Cassini discovery)

`scene/examples/saturn_voyage.json` drives three classic Cassini-style shots: oblique full-disc with rings, low-elevation ring skim showing the ring shadow stretched across the planet, and a polar dive that frames the north-pole hexagon. The renderer is general enough to also produce Uranus-tilted or Neptune-blue variants by swapping the band palette and ring radii.

In [ ]:
import os, time
from base64 import b64encode
from IPython.display import HTML, display

from scene.manifest import load_manifest
import render.adapters  # registers saturn_class
from director.graph import GraphRunner

os.makedirs('outputs', exist_ok=True)

sat_graph = load_manifest('scene/examples/saturn_voyage.json')

# Fast-mode downsample so the cell finishes in a few minutes on a T4.
for s in sat_graph.shots:
    s.resolution = (640, 360)
    s.duration_frames = max(24, s.duration_frames // 3)
for tr in sat_graph.transitions:
    tr.duration_frames = max(4, tr.duration_frames // 3)

t0 = time.time()
GraphRunner(sat_graph, 'outputs/saturn_voyage.mp4').run()
print(f'wrote outputs/saturn_voyage.mp4 in {time.time() - t0:.1f}s')

display(HTML(_video_html('outputs/saturn_voyage.mp4', width=820)))

## 14. Stellar surface — granulation, sunspots, prominences

`stellar_surface` renders a star with:

- **Limb darkening** via the linear law `I(μ) = I₀(1 − u + uμ)`
- **Two-scale Worley granulation** — granules + supergranules from the noise toolkit
- **Parametric sunspots**: list of `[lat, lon, radius_deg, darkness]`. Darker umbra + brighter penumbra ring
- **Halpha prominences**: pink plasma arcs anchored at the limb, rendered in a corona-shell volumetric pass
- **Effective-temperature color** from a Planck blackbody approximation — vary `T_eff` from 1500K (brown dwarf) through 5800K (Sun) to 32000K (O-star)
- **Procedural fine-scale detail** in world space (footprint-AA), so close-ups don't pixelate

`scene/examples/stellar_close.json` drives three contrasting stars: a sun-like active region with three sunspots and two prominences, a cool M-dwarf with two large starspots, and a hot O-star with no spots but big polar prominences.

In [ ]:
import os, time
from base64 import b64encode
from IPython.display import HTML, display

from scene.manifest import load_manifest
import render.adapters  # registers stellar_surface
from director.graph import GraphRunner

os.makedirs('outputs', exist_ok=True)

star_graph = load_manifest('scene/examples/stellar_close.json')

# Fast-mode downsample so the cell finishes quickly on a T4.
for s in star_graph.shots:
    s.resolution = (640, 360)
    s.duration_frames = max(24, s.duration_frames // 3)
for tr in star_graph.transitions:
    tr.duration_frames = max(4, tr.duration_frames // 3)

t0 = time.time()
GraphRunner(star_graph, 'outputs/stellar_close.mp4').run()
print(f'wrote outputs/stellar_close.mp4 in {time.time() - t0:.1f}s')

display(HTML(_video_html('outputs/stellar_close.mp4', width=820)))

## 15. Studio: artifact store + metrics critic

Every `GraphRunner.run()` can now be steered into a self-contained, addressable artifact directory: the manifest, provenance (git SHA, library versions), the video, per-shot keyframes (`.npy` for exact metrics + tonemapped `.png` for humans and VLMs), and a metrics sidecar. A deterministic `MetricsCritic` then scores any artifact for temporal coherence, palette adherence, dynamic range, edge density, and common failure modes (NaN runs, frozen frames, near-degenerate flats).

The keyframe strip rendered below is what a future local-VLM critic will receive as its visual input.

In [ ]:
from studio import ArtifactStore, MetricsCritic
from pathlib import Path
from IPython.display import HTML, display
from base64 import b64encode

store = ArtifactStore("renders")
art_path = GraphRunner(graph, "_unused.mp4", artifact_store=store, artifact_label="baseline").run()
print("artifact:", art_path)

result = MetricsCritic().evaluate_artifact(art_path)
agg = result.aggregate
print(f"\naggregate composite score : {agg['composite_score']:.3f}")
print(f"worst shot                : {agg['worst_shot']['id']}  ({agg['worst_shot']['composite_score']:.3f})")
print()
print(f"  {'shot':<18s} {'score':>6s} {'palette_d':>10s} {'temporal':>10s} {'edge':>8s} {'notes'}")
for sid, m in result.shots.items():
    notes = "; ".join(m.notes) if m.notes else "-"
    print(f"  {sid:<18s} {m.composite_score:>6.3f} {m.palette_distance:>10.4f} {m.temporal_abs_diff:>10.5f} {m.edge_density:>8.4f} {notes}")


def show_keyframes(artifact_dir, shot_id, height=110):
    kdir = Path(artifact_dir) / "shots" / shot_id / "keyframes"
    pngs = sorted(kdir.glob("*.png"))
    if not pngs:
        return HTML(f"<p>no keyframes for {shot_id}</p>")
    imgs = []
    for p in pngs:
        b64 = b64encode(p.read_bytes()).decode()
        imgs.append(f'<img src="data:image/png;base64,{b64}" style="height:{height}px; margin:2px;" />')
    return HTML(
        f'<div style="margin:6px 0;"><strong>{shot_id}</strong>'
        f'<div style="display:flex;flex-wrap:wrap;">{"".join(imgs)}</div></div>'
    )

for sid in result.shots:
    display(show_keyframes(art_path, sid))

## 16. Producer + Session loop (closing the critic loop)

The critic earns its keep only when something *acts* on its scores. A `MutationSpace` declares which knobs the loop may turn; a `Producer` picks one knob per iteration and perturbs it within bounds; a `Session` renders each candidate at proxy resolution, scores it via the critic, caches duplicates, and tracks the best so far.

The cell below explores the hand-authored two-shot graph from section 9 (`my_graph`). Expect the first proposal to pay the Kerr JIT cost; subsequent ones run much faster. The proxy keeps each render under 30 s on a T4.

In [ ]:
from studio import (
    ArtifactStore, MetricsCritic, MutationSpace, RandomProducer,
    Session, shot_param, shot_palette, transition_kind,
)

space = MutationSpace([
    shot_param("kerr",   "spin",            kind="float",  bounds=(0.2, 0.999), perturb_scale=0.25),
    shot_param("kerr",   "inclination_deg", kind="float",  bounds=(45.0, 89.0), perturb_scale=0.2),
    shot_param("kerr",   "disk_outer",      kind="float",  bounds=(8.0, 22.0),  perturb_scale=0.2),
    shot_palette("kerr",   choices=("trumbull_2001", "hubble_sii_ha_oiii", "jwst_nircam")),
    shot_palette("tunnel", choices=("trumbull_2001", "jwst_nircam")),
    transition_kind("kerr", "tunnel", choices=("crossfade", "slitscan", "match_cut")),
])

session = Session(
    base_graph=my_graph,
    store=ArtifactStore("renders/session"),
    producer=RandomProducer(my_graph, space, seed=42),
    critic=MetricsCritic(),
    proxy={"resolution": (320, 180), "duration_scale": 0.5},
)

def _on(att):
    tag = "  (cached)" if att.cached else ""
    desc = f"{att.mutation['param']}: {att.mutation['from']} -> {att.mutation['to']}"
    print(f"  iter {att.iteration:>2}  score {att.score:.3f}  {desc}{tag}")

print("Exploring at 320x180 proxy:")
session.explore(budget=6, label_prefix="lab1", on_attempt=_on)
print()

best = session.best()
print(f"Winner: iter {best.iteration}  score {best.score:.3f}")
print(f"Artifact: {best.artifact_path}")
log_path = session.save("renders/session_log.json")
print(f"Session log written to {log_path}")

from IPython.display import display
for shot in my_graph.shots:
    display(show_keyframes(best.artifact_path, shot.id))

## 17. Where to go next

- **Try the unused topology variants.** Swap nebula `topology` to `veil`, `orion`, or `pleiades`; vary `stellar_surface` `T_eff` to scan from brown dwarf to O-star; flip Saturn ring radii to make Uranus-tilted or Neptune-blue ringed giants. Each is a parameter, not a new renderer.
- **Tune any manifest in the Studio loop.** Build a `MutationSpace` over a renderer's knobs and let the producer sample plausible variants -- the alien-yet-real criterion.
- **Render the winning manifest at full quality.** `session.render_best_at_full_quality(artifact_label="master")`.
- **Try the epsilon-greedy producer.** Swap `RandomProducer` for `EpsilonGreedyProducer` to bias toward perturbations of the current best.
- **Bump up quality.** Comment out the FAST_MODE blocks in sections 6, 10, 11, 12, 13, and 14 for the full-resolution masters.
- **Add a local multimodal critic.** Next on the roadmap: a Qwen2-VL-2B (or Phi-3.5-vision) wrapper that runs on the Colab GPU. Reads the keyframes + manifest, returns structured per-shot scores plus one actionable suggestion. Slot it in via `Session(critic=...)`. No API keys, no rate limits, no network dependency.
- **Add a renderer.** Implement a subclass of `render.core.Renderer`, decorate it with `@register_renderer('your_name')`, and reference it from a Shot. Next on the roadmap: terrain with ridged fBM + hydraulic erosion, cellular-automata creatures (Lenia / Karl Sims), exoplanet-class atmospheres.
- **Inspect any artifact from the shell.** `python -m studio.cli list`, `python -m studio.cli evaluate <path>`, `python -m studio.cli show <path>`.